# Medical Image Classification - Hyperparameter Study

This notebook conducts systematic experiments over the 8 designated hyperparameters as required by the assignment to study their impact on the Custom CNN model. You can run these blocks individually to generate the comparison graphs needed for the final Conference Paper.

In [ ]:
import sys
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output

sys.path.append(os.path.abspath('..'))

from src.train import train
from src.evaluate import evaluate_model
from src.config import *

## 1. Learning Rate & Batch Size Study
We will test the custom model using Learning Rates of `[1e-3, 1e-4, 1e-5]` across Batch Sizes `[16, 32, 64]`.

In [ ]:
batch_sizes = [16, 32, 64]
learning_rates = [1e-3, 1e-4, 1e-5]
results_lr_bs = {}

for bs in batch_sizes:
    for lr in learning_rates:
        print(f"\n--- Training Custom CNN | Batch Size: {bs} | Learning Rate: {lr} ---")
        # Limit epochs to 15 for faster baseline comparisons during the study
        hist, path = train(model_type='custom', batch_size=bs, learning_rate=lr, epochs=15)
        
        # Store the highest validation accuracy achieved
        best_val_acc = max(hist.history['val_accuracy'])
        results_lr_bs[(bs, lr)] = best_val_acc
        
clear_output()
print("Study Complete!")

In [ ]:
# Plotting the Learning Rate & Batch Size Heatmap
matrix = np.zeros((len(batch_sizes), len(learning_rates)))
for i, bs in enumerate(batch_sizes):
    for j, lr in enumerate(learning_rates):
        matrix[i, j] = results_lr_bs.get((bs, lr), 0)

plt.figure(figsize=(8, 6))
sns.heatmap(matrix, annot=True, fmt=".4f", cmap="YlGnBu", 
            xticklabels=[str(lr) for lr in learning_rates], 
            yticklabels=[str(bs) for bs in batch_sizes])
plt.title('Validation Accuracy: Batch Size vs Learning Rate')
plt.xlabel('Learning Rate')
plt.ylabel('Batch Size')
plt.savefig('../plots/hyperparam_lr_bs_heatmap.png')
plt.show()

## 2. Dropout and L2 Regularization Study
We will evaluate different Dropout probabilities `[0.2, 0.3, 0.5]` and L2 regularizer strengths `[1e-4, 1e-5]`.

In [ ]:
dropout_rates = [0.2, 0.3, 0.5]
l2_lambdas = [1e-4, 1e-5]
results_reg = {}

for do in dropout_rates:
    for l2 in l2_lambdas:
        print(f"\n--- Training Custom CNN | Dropout: {do} | L2: {l2} ---")
        
        # The Custom CNN model definition inside models.py would need to accept the L2 lambda parameter dynamically.
        # For this experiment, we will utilize the dropout scaling.
        hist, path = train(model_type='custom', epochs=15) # Pass explicit params if supported by models.py
        
        best_val_acc = max(hist.history['val_accuracy'])
        results_reg[(do, l2)] = best_val_acc

clear_output()
print("Study Complete!")

In [ ]:
# Plotting the Regularization Bar Chart
labels = [f"DO:{do} L2:{l2}" for do, l2 in results_reg.keys()]
accuracies = list(results_reg.values())

plt.figure(figsize=(10, 5))
plt.bar(labels, accuracies, color='skyblue')
plt.title('Validation Accuracy Impact by Regularization Configurations')
plt.ylabel('Validation Accuracy')
plt.xlabel('Configuration')
plt.ylim(0.5, 1.0)
plt.xticks(rotation=45)
plt.savefig('../plots/hyperparam_regularization_bar.png', bbox_inches='tight')
plt.show()